# CNN Leanring with CIFAR-10 data set

In [131]:
import torch
import torch.nn as nn
import torch.nn.functional as f
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# finding batch level mean and std valus of the CIFAR10 images

In [62]:
print(torch.backends.mps.is_available())
print(torch.backends.mps.is_built())
print(torch.device("mps"))

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

print(device)

True
True
mps
mps


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)




In [32]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    batch_size=64
)

In [39]:


mean = torch.zeros(3)
std = torch.zeros(3)

for images, _ in train_loader:
    # print(images.shape) # expected [batch, channels, height, width]

    for c in range(3):
        mean[c] += images[:, c, :, :].mean()
        std[c] += images[:, c, :, :].std()

mean /= len(train_loader)
std /= len(train_loader)

print(len(train_loader))
print(f"mean: {mean}")
print(f"std: {std}")


782
mean: tensor([0.4915, 0.4822, 0.4466])
std: tensor([0.2463, 0.2428, 0.2607])


# loading data with mean and std calculated

In [118]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4915, 0.4822, 0.4466),
        [0.2463, 0.2428, 0.2607]
    )
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4915, 0.4822, 0.4466),
        [0.2463, 0.2428, 0.2607]
    )
])

In [132]:
train_dataset_aug = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=train_transform
)

train_dataset_clear = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=test_transform
)

test_dataset = datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=test_transform
)


generator = torch.Generator().manual_seed(42)

indices = torch.randperm(
    len(train_dataset_aug),
    generator=generator
).tolist()

train_indices = indices[:45000]
val_indices = indices[45000:]

train_subset = Subset(
    train_dataset_aug,
    train_indices
)

val_subset = Subset(
    train_dataset_clear,
    val_indices
)

train_loader = DataLoader(
    train_subset,
    batch_size=64,
    shuffle=True,
)

val_loader = DataLoader(
    val_subset,
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

/Users/raj/Documents/Corpnce AI/practice/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


# Buliding base model

In [91]:
# Building base model

class CNN_base(nn.Module):
    def __init__(self):
        super().__init__()

        # image size (3, 32, 32)  - (32, 30, 30) # 32 features maps, each of size 26, 26 as kernal is 3
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3)

        self.conv2 = nn.Conv2d(32, 65, kernel_size=3)

        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # print(f"input shape = {x.shape}")

        x = self.conv1(x)
        # print(f"after conv1: {x.shape}")
        x = f.relu(x) # shape does not change


        
        x = f.max_pool2d(x, 2) # compress to using 2 * 2 
        # print("After pool1:", x.shape)

        x = self.conv2(x)
        # print(f"after conv2 {x.shape}")
        x = f.relu(x)
        x = f.max_pool2d(x, 2) # compress by using 2* 2 kernel
        # print(f"after pool2: ", x.shape)

        x = torch.flatten(x, 1)
        # print(f"after flatter: ", x.shape)

        x = f.relu(self.fc1(x))
        x = self.fc2(x)

        return x
    

In [ ]:
# Building base model

class CNN_padding(nn.Module):
    def __init__(self):
        super().__init__()

        # image size (3, 32, 32)  - (32, 30, 30) # 32 features maps, each of size 26, 26 as kernal is 3
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # print(f"input shape = {x.shape}")

        x = self.conv1(x)
        # print(f"after conv1: {x.shape}")
        x = f.relu(x) # shape does not change
        
        x = f.max_pool2d(x, 2) # compress to using 2 * 2 
        # print("After pool1:", x.shape)

        x = self.conv2(x)
        # print(f"after conv2 {x.shape}")
        x = f.relu(x)
        x = f.max_pool2d(x, 2) # compress by using 2* 2 kernel
        # print(f"after pool2: ", x.shape)

        x = torch.flatten(x, 1)
        # print(f"after flatter: ", x.shape)

        x = f.relu(self.fc1(x))
        x = self.fc2(x)

        return x
    

In [101]:
# Building base model

class CNN_padding_stride(nn.Module):
    def __init__(self):
        super().__init__()

        # image size (3, 32, 32)  - (32, 30, 30) # 32 features maps, each of size 26, 26 as kernal is 3
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1, stride=2)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=2)

        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # print(f"input shape = {x.shape}")

        x = self.conv1(x)
        # print(f"after conv1: {x.shape}")
        x = f.relu(x) # shape does not change
        
        # x = f.max_pool2d(x, 2) # compress to using 2 * 2 
        # print("After pool1:", x.shape)

        x = self.conv2(x)
        # print(f"after conv2 {x.shape}")
        x = f.relu(x)
        # x = f.max_pool2d(x, 2) # compress by using 2* 2 kernel
        # print(f"after pool2: ", x.shape)

        x = torch.flatten(x, 1)
        # print(f"after flatter: ", x.shape)

        x = f.relu(self.fc1(x))
        x = self.fc2(x)

        return x
    

In [120]:
# Building base model

class CNN_conv4_padd1(nn.Module):
    def __init__(self):
        super().__init__()

        # image size (3, 32, 32)  - (32, 30, 30) # 32 features maps, each of size 26, 26 as kernal is 3
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)

        # (32, 15, 15) -> 64, 13, 13
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)

        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(32)
        self.bn3 = nn.BatchNorm2d(64)
        self.bn4 = nn.BatchNorm2d(64)

        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # print(f"input shape = {x.shape}")

        x = self.conv1(x)
        x = self.bn1(x)
        # print(f"after conv1: {x.shape}")
        x = f.relu(x) # shape does not change

        x = self.conv2(x)
        x = self.bn2(x)

        # print(f"after conv1: {x.shape}")
        x = f.relu(x) # shape does not change
        
        x = f.max_pool2d(x, 2) # compress to using 2 * 2 
        # print("After pool1:", x.shape)

        x = self.conv3(x)
        x = self.bn3(x)
        # print(f"after conv2 {x.shape}")
        x = f.relu(x)
        # print(f"after pool2: ", x.shape)

        x = self.conv4(x)
        x = self.bn4(x)
        # print(f"after conv2 {x.shape}")
        x = f.relu(x)
        x = f.max_pool2d(x, 2) # compress by using 2* 2 kernel
        # print(f"after pool2: ", x.shape)

        x = torch.flatten(x, 1)
        # print(f"after flatter: ", x.shape)

        x = f.relu(self.fc1(x))
        x = self.fc2(x)

        return x
    

In [109]:
# Building base model

class CNN_conv4_padd1_drop(nn.Module):
    def __init__(self):
        super().__init__()

        # image size (3, 32, 32)  - (32, 30, 30) # 32 features maps, each of size 26, 26 as kernal is 3
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)

        # (32, 15, 15) -> 64, 13, 13
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)

        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(32)
        self.bn3 = nn.BatchNorm2d(64)
        self.bn4 = nn.BatchNorm2d(64)

        self.dropout = nn.Dropout(p=0.2)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # print(f"input shape = {x.shape}")

        x = self.conv1(x)
        x = self.bn1(x)
        # print(f"after conv1: {x.shape}")
        x = f.relu(x) # shape does not change

        x = self.conv2(x)
        x = self.bn2(x)

        # print(f"after conv1: {x.shape}")
        x = f.relu(x) # shape does not change
        
        x = f.max_pool2d(x, 2) # compress to using 2 * 2 
        # print("After pool1:", x.shape)

        x = self.conv3(x)
        x = self.bn3(x)
        # print(f"after conv2 {x.shape}")
        x = f.relu(x)
        # print(f"after pool2: ", x.shape)

        x = self.conv4(x)
        x = self.bn4(x)
        # print(f"after conv2 {x.shape}")
        x = f.relu(x)
        x = f.max_pool2d(x, 2) # compress by using 2* 2 kernel
        # print(f"after pool2: ", x.shape)

        x = torch.flatten(x, 1)
        # print(f"after flatter: ", x.shape)

        x = f.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x
    

In [ ]:
model = CNN_conv4_padd1()

model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001) # model class know all the paramenters in the order of execution
criterion = nn.CrossEntropyLoss()

correct = 0
total = 0
model.train()

for epoch in range(20):
    for batch_idx, (data, target) in enumerate(train_loader):

        data = data.to(device)
        target = target.to(device)
        
        optimizer.zero_grad()

        output = model(data)
        loss = criterion(output, target)

        loss.backward()
        optimizer.step()

        if batch_idx % 100 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item()}")



model.eval()
correct = 0
total = 0

with torch.no_grad():
    for data, target in train_loader:
        data = data.to(device)
        target = target.to(device)

        output = model(data)

        predicted = torch.argmax(output, dim=1)

        total += target.size(0)
        correct += (predicted == target).sum().item()

print(f"train accuracy = {correct/total}")


correct = 0
total = 0

model.eval()

with torch.no_grad():
    for data, target in test_loader:
        data = data.to(device)
        target = target.to(device)

        output = model(data)
        _, predicted = torch.max(output, 1)

        # print(f"target shape = {target.shape} and zero pos = {target.size(0)}")
        total += target.size(0)
        correct += (predicted == target).sum().item()

print(f"test accuracy = {correct/total}")

Epoch 0, Loss: 2.334087371826172
Epoch 0, Loss: 1.579136848449707
Epoch 0, Loss: 1.5743424892425537
Epoch 0, Loss: 1.4332185983657837
Epoch 0, Loss: 1.207667350769043
Epoch 0, Loss: 1.3649983406066895
Epoch 0, Loss: 1.3555091619491577
Epoch 0, Loss: 0.9063233137130737
Epoch 1, Loss: 0.9569613933563232
Epoch 1, Loss: 1.182811975479126
Epoch 1, Loss: 0.9601558446884155
Epoch 1, Loss: 1.0760083198547363
Epoch 1, Loss: 1.1686608791351318
Epoch 1, Loss: 1.3430395126342773
Epoch 1, Loss: 1.265999436378479
Epoch 1, Loss: 1.0057016611099243
Epoch 2, Loss: 0.747496485710144
Epoch 2, Loss: 0.9606133699417114
Epoch 2, Loss: 1.008391261100769
Epoch 2, Loss: 0.7077183723449707
Epoch 2, Loss: 0.706655740737915
Epoch 2, Loss: 0.7846246361732483
Epoch 2, Loss: 1.165448546409607
Epoch 2, Loss: 0.9535355567932129
Epoch 3, Loss: 0.7766730785369873
Epoch 3, Loss: 0.7066645622253418
Epoch 3, Loss: 0.7672642469406128
Epoch 3, Loss: 0.7897138595581055
Epoch 3, Loss: 1.1290782690048218
Epoch 3, Loss: 0.838263

In [ ]:
# code before validation changes

for epoch in range(20):
    for batch_idx, (data, target) in enumerate(train_loader):

        data = data.to(device)
        target = target.to(device)
        
        optimizer.zero_grad()

        output = model(data)
        loss = criterion(output, target)

        loss.backward()
        optimizer.step()

        if batch_idx % 100 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item()}")



model.eval()
correct = 0
total = 0

with torch.no_grad():
    for data, target in train_loader:
        data = data.to(device)
        target = target.to(device)

        output = model(data)

        predicted = torch.argmax(output, dim=1)

        total += target.size(0)
        correct += (predicted == target).sum().item()

print(f"train accuracy = {correct/total}")


correct = 0
total = 0

model.eval()

with torch.no_grad():
    for data, target in test_loader:
        data = data.to(device)
        target = target.to(device)

        output = model(data)
        _, predicted = torch.max(output, 1)

        # print(f"target shape = {target.shape} and zero pos = {target.size(0)}")
        total += target.size(0)
        correct += (predicted == target).sum().item()

print(f"test accuracy = {correct/total}")

In [135]:
model = CNN_conv4_padd1()

model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001) # model class know all the paramenters in the order of execution
criterion = nn.CrossEntropyLoss()

train_losses = []
train_accuracies = []

val_losses = []
val_accuracies = []

correct = 0
total = 0
# model.train()

num_epochs = 20

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for data, target in train_loader:

        data = data.to(device)
        target = target.to(device)

        optimizer.zero_grad()

        output = model(data)

        loss = criterion(output, target)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * data.size(0)

        predicted = torch.argmax(output, dim=1)

        total += target.size(0)

        correct += (predicted == target).sum().item()

    epoch_train_loss = running_loss / total
    epoch_train_accuracy = correct/total

    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_accuracy)



    # validation

    model.eval()
    running_loss = 0.0
    correct = 0.0
    total = 0.0

    with torch.no_grad():
        for data, target in val_loader:

            data = data.to(device)
            target = target.to(device)

            output = model(data)

            loss = criterion(output, target)

            running_loss += loss.item() * data.size(0)

            predicted = torch.argmax(output, dim=1)

            total += target.size(0)
            correct += (predicted == target).sum().item()

        epoch_val_loss = running_loss / total
        epoch_val_accuracy = correct / total

        val_losses.append(epoch_val_loss)
        val_accuracies.append(epoch_val_accuracy)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f" train loss: {epoch_train_loss} "
        f" train acc: {epoch_train_accuracy} "
        f" val loss: {epoch_val_loss} "
        f" val acc: {epoch_val_accuracy}")




Epoch [1/20]  train loss: 1.458501484298706  train acc: 0.4656888888888889  val loss: 1.0876131954193116  val acc: 0.6112
Epoch [2/20]  train loss: 1.081535740439097  train acc: 0.6114  val loss: 1.1648729983329773  val acc: 0.617
Epoch [3/20]  train loss: 0.9206096514383951  train acc: 0.6728888888888889  val loss: 0.8453047707557678  val acc: 0.696
Epoch [4/20]  train loss: 0.838474419148763  train acc: 0.7035333333333333  val loss: 0.8802908935546875  val acc: 0.6988
Epoch [5/20]  train loss: 0.7827438684357537  train acc: 0.7234888888888888  val loss: 0.77658586063385  val acc: 0.7236
Epoch [6/20]  train loss: 0.7426939591513739  train acc: 0.7388  val loss: 0.6708410134315491  val acc: 0.7666
Epoch [7/20]  train loss: 0.7065038679334853  train acc: 0.7532666666666666  val loss: 0.6498520373344422  val acc: 0.764
Epoch [8/20]  train loss: 0.6776524328496721  train acc: 0.7622  val loss: 0.716821651172638  val acc: 0.7578
Epoch [9/20]  train loss: 0.6494247268570794  train acc: 0.77

- basic run: train accuracy = 0.7521. accuracy = 0.6922. (2 convolution layer 2 fcnn layer )
- run2 : train accuracy = 0.70526, accuracy = 0.669 (3 convolution layer, 2 fcnn layer)
- run3: train accuracy = 0.78678, test accuracy = 0.7215 (4 convolution layers - conv1-conv2-pool-conv3-conv4-pool + padding=1)
- run4: train accuracy = 0.78038, test accuracy = 0.7089 (2 convolution, 2 fc nn + padding 1)
- run5: train accuracy = 0.7529, test accuracy = 0.6642 (2 conv, 2 fcnn, stride=2)
- run6: train accuracy = 0.81146, test accuracy = 0.7619 (4 convolution layers - conv1-conv2-pool-conv3-conv4-pool + padding=1 + batchnorm2d)
- run7: train accuracy = 0.64636, test accuracy = 0.6287 (4 convolution layers - conv1-conv2-pool-conv3-conv4-pool + padding=1 + batchnorm2d, dropout=0.5)
- run8: train accuracy = 0.7716, test accuracy = 0.7359 (4 convolution layers - conv1-conv2-pool-conv3-conv4-pool + padding=1 + batchnorm2d, dropout=0.2)
- run9: train accuracy = 0.6951 test accuracy = 0.7187 (4 convolution layers - conv1-conv2-pool-conv3-conv4-pool + padding=1 + batchnorm2d + augmentation)
- run10: train accuracy = 0.7865, test accuracy = 0.7845 (4 convolution layers - conv1-conv2-pool-conv3-conv4-pool + padding=1 + batchnorm2d + augmentation + 10 epochs)
- run11: train acc: 0.8274666666666667 val acc: 0.814, (4 convolution layers - conv1-conv2-pool-conv3-conv4-pool + padding=1 + batchnorm2d + augmentation + 20 epochs, validation is done on training data)